<a href="https://colab.research.google.com/github/wan-chtvlv/spatial-coastal-ses/blob/main/B2_2_CopernicusMarine_current_velocity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

I asked Gemini "How to use ocean hydrodynamic data on colab" and adjusted the code from there.

Copernicus webpage: https://marine.copernicus.eu/

Their data store: https://data.marine.copernicus.eu/products?pk_vid=d9a5c3740f312f9117915160884d3b3f

Article on ocean's current: https://marine.copernicus.eu/news/ocean-currents-earths-invisible-engine. From the last set of images in the article, they suggest using "the daily sea water velocity dataset: GLOBAL_ANALYSISFORECAST_PHY_001_024. Credit: European Union, Copernicus Marine Service Data (2026), © Mercator Ocean.

Their python library: https://pypi.org/project/copernicusmarine/

In [ ]:
!pip install xarray copernicusmarine cftime cartopy nc-time-axis ee StringIO

ERROR: Could not find a version that satisfies the requirement StringIO (from versions: none)
ERROR: No matching distribution found for StringIO


In [ ]:
import copernicusmarine
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
# import ee

Data fetching can be done in 2 ways

1) directly downloaded from Copernicus
2) fetch through Google Earth Engine

In [ ]:
# Code suggested by Gemini
# ------------------------------------------------------------------------
# Log in using your Copernicus credentials see txt file on computer
copernicusmarine.login(username="email", password="password")

# https://data.marine.copernicus.eu/product/GLOBAL_ANALYSISFORECAST_PHY_001_024/description
# https://documentation.marine.copernicus.eu/PUM/CMEMS-GLO-PUM-001-024.pdf
# Needs to read through this documentation to figure out which dataset makes the most sense.
dataset_id = "GLOBAL_ANALYSISFORECAST_PHY_001_024"

copernicusmarine.subset(
    dataset_id=dataset_id,
    variables=["uo", "vo"],
    start_datetime="2026-01-01T00:00:00",
    end_datetime="2026-01-05T00:00:00",
    minimum_longitude=-75.0,
    maximum_longitude=-70.0,
    minimum_latitude=35.0,
    maximum_latitude=40.0,
    minimum_depth=0,
    maximum_depth=5,
    output_directory="./data",
    output_filename="ocean_currents.nc"
)

In [ ]:
# Cannot install ee yet, it throws an error for 'No module named StringIO'
import ee
ee.Authenticate()
ee.Initialize()

In [ ]:
# Code suggested by Gemini
# Load and analyze the data
# ------------------------------------------------------------------------
# 1. Load the NetCDF file
ds = xr.open_dataset("./data/ocean_currents.nc")
print(ds) # This displays metadata, variables, and dimensions (Time, Depth, Lat, Lon)

# 2. Extract a specific time slice at the surface level
surface_currents = ds.isel(time=0, depth=0)

# 3. Calculate Eastward/Northward magnitude (velocity speed)
# 'uo' = zonal velocity (East/West), 'vo' = meridional velocity (North/South)
current_speed = (surface_currents.uo**2 + surface_currents.vo**2)**0.5

# 4. Plot using Cartopy
plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()

# Draw the hydrodynamic speeds
current_speed.plot(ax=ax, x="longitude", y="latitude", cmap="YlGnBu", cbar_kwargs={'label': 'Current Speed (m/s)'})

plt.title("Ocean Surface Current Velocities")
plt.show()